In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import time

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report,
    confusion_matrix, roc_auc_score
)

from xgboost import XGBClassifier

print("All libraries imported successfully!")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("XGBoost:", XGBClassifier.__module__)

All libraries imported successfully!
Pandas: 3.0.6
NumPy: 2.5.4
XGBoost: xgboost.sklearn


In [3]:
# Dataset path
DATA_PATH = Path("../data/processed/cicids2017_feature_engineered.csv")

print("Checking dataset path:", DATA_PATH.resolve())

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}")

# Load dataset
df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Checking dataset path: D:\ThreatTrace\data\processed\cicids2017_feature_engineered.csv

Dataset loaded successfully!
Dataset shape: (2517883, 66)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [4]:
# Create binary target
df["Binary_Label"] = np.where(
    df["Label"].str.strip() == "BENIGN",
    0,
    1
)

# Check binary class distribution
binary_counts = df["Binary_Label"].value_counts().sort_index()

print("BINARY CLASS DISTRIBUTION")
print("=" * 40)
print("BENIGN (0):", binary_counts.get(0, 0))
print("ATTACK (1):", binary_counts.get(1, 0))

print("\nClass percentages:")
display(
    (df["Binary_Label"].value_counts(normalize=True) * 100)
    .sort_index()
    .rename(index={0: "BENIGN", 1: "ATTACK"})
    .to_frame("Percentage")
)

print("\nTotal binary classes:", df["Binary_Label"].nunique())
print("Total records:", len(df))

BINARY CLASS DISTRIBUTION
BENIGN (0): 2092335
ATTACK (1): 425548

Class percentages:


,Percentage
Binary_Label,
BENIGN,83.098976
ATTACK,16.901024



Total binary classes: 2
Total records: 2517883


In [5]:
# Create binary target
df["Binary_Label"] = np.where(
    df["Label"].str.strip() == "BENIGN",
    0,
    1
)

# Check binary class distribution
binary_counts = df["Binary_Label"].value_counts().sort_index()

print("BINARY CLASS DISTRIBUTION")
print("=" * 40)
print("BENIGN (0):", binary_counts.get(0, 0))
print("ATTACK (1):", binary_counts.get(1, 0))

print("\nClass percentages:")
display(
    (df["Binary_Label"].value_counts(normalize=True) * 100)
    .sort_index()
    .rename(index={0: "BENIGN", 1: "ATTACK"})
    .to_frame("Percentage")
)

print("\nTotal binary classes:", df["Binary_Label"].nunique())
print("Total records:", len(df))

BINARY CLASS DISTRIBUTION
BENIGN (0): 2092335
ATTACK (1): 425548

Class percentages:


,Percentage
Binary_Label,
BENIGN,83.098976
ATTACK,16.901024



Total binary classes: 2
Total records: 2517883


In [6]:
# Separate features and target
feature_columns = [
    col for col in df.columns
    if col not in ["Label", "Binary_Label"]
]

X_check = df[feature_columns]

print("BINARY DATASET VALIDATION")
print("=" * 45)

print("Total features:", len(feature_columns))
print("Total records:", len(df))

print("\nMissing values:", X_check.isnull().sum().sum())

print("Infinite values:",
      np.isinf(X_check.select_dtypes(include=np.number).to_numpy()).sum())

print("Non-numeric columns:",
      X_check.select_dtypes(exclude=np.number).columns.tolist())

print("\nTarget values:", sorted(df["Binary_Label"].unique()))

print("\nFeature columns:")
print(feature_columns)

BINARY DATASET VALIDATION
Total features: 65
Total records: 2517883

Missing values: 0
Infinite values: 0
Non-numeric columns: []

Target values: [np.int64(0), np.int64(1)]

Feature columns:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Le

In [7]:
# Separate features and target
X = df[feature_columns]
y = df["Binary_Label"].to_numpy()

# Create row indices to reduce memory usage
indices = np.arange(len(df))

# First split: 70% training + 30% temporary
train_val_idx, test_idx = train_test_split(
    indices,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Second split: 70% training + 15% validation
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.17647,
    random_state=42,
    stratify=y[train_val_idx]
)

print("TRAINING RECORDS:", len(train_idx))
print("VALIDATION RECORDS:", len(val_idx))
print("TEST RECORDS:", len(test_idx))

print("\nTraining class distribution:")
print(pd.Series(y[train_idx]).value_counts().sort_index())

print("\nValidation class distribution:")
print(pd.Series(y[val_idx]).value_counts().sort_index())

print("\nTesting class distribution:")
print(pd.Series(y[test_idx]).value_counts().sort_index())

TRAINING RECORDS: 1762518
VALIDATION RECORDS: 377682
TEST RECORDS: 377683

Training class distribution:
0    1464634
1     297884
Name: count, dtype: int64

Validation class distribution:
0    313850
1     63832
Name: count, dtype: int64

Testing class distribution:
0    313851
1     63832
Name: count, dtype: int64


In [8]:
# Select 600,000 stratified training records
train_sample_idx, _ = train_test_split(
    train_idx,
    train_size=600000,
    random_state=42,
    stratify=y[train_idx]
)

# Convert features to float32
X_train = df.iloc[train_sample_idx][feature_columns].to_numpy(
    dtype=np.float32
)

X_val = df.iloc[val_idx][feature_columns].to_numpy(
    dtype=np.float32
)

X_test = df.iloc[test_idx][feature_columns].to_numpy(
    dtype=np.float32
)

# Target arrays
y_train = y[train_sample_idx]
y_val = y[val_idx]
y_test = y[test_idx]

print("DATA PREPARATION COMPLETE")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("\nTraining class distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTraining memory:",
      round(X_train.nbytes / (1024**2), 2), "MB")

print("Validation memory:",
      round(X_val.nbytes / (1024**2), 2), "MB")

print("Testing memory:",
      round(X_test.nbytes / (1024**2), 2), "MB")

DATA PREPARATION COMPLETE
Training: (600000, 65)
Validation: (377682, 65)
Testing: (377683, 65)

Training class distribution:
0    498594
1    101406
Name: count, dtype: int64

Training memory: 148.77 MB
Validation memory: 93.65 MB
Testing memory: 93.65 MB


In [9]:
from sklearn.utils.class_weight import compute_sample_weight

# Calculate balanced sample weights
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

print("BINARY CLASS WEIGHTS")
print("=" * 40)

for class_id, class_name in [(0, "BENIGN"), (1, "ATTACK")]:
    print(
        f"{class_name}:",
        round(sample_weights[y_train == class_id][0], 4)
    )

print("\nSample weights shape:", sample_weights.shape)
print("Minimum weight:", round(sample_weights.min(), 4))
print("Maximum weight:", round(sample_weights.max(), 4))
print("Average weight:", round(sample_weights.mean(), 4))

BINARY CLASS WEIGHTS
BENIGN: 0.6017
ATTACK: 2.9584

Sample weights shape: (600000,)
Minimum weight: 0.6017
Maximum weight: 2.9584
Average weight: 1.0


In [10]:
xgb_binary = XGBClassifier(
    objective="binary:logistic",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    reg_alpha=0.1,
    tree_method="hist",
    max_bin=256,
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=4
)

print("XGBoost Binary model configured successfully!")
print("Objective: binary:logistic")
print("Maximum estimators: 500")
print("Maximum depth: 6")
print("Tree method: hist")

XGBoost Binary model configured successfully!
Objective: binary:logistic
Maximum estimators: 500
Maximum depth: 6
Tree method: hist


In [11]:
print("Starting XGBoost Binary training...")

start_time = time.time()

xgb_binary.fit(
    X_train,
    y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val, y_val)],
    verbose=25
)

training_time = time.time() - start_time

print("\nBINARY TRAINING COMPLETED")
print("=" * 40)
print("Training time:",
      round(training_time / 60, 2), "minutes")

print("Best iteration:", xgb_binary.best_iteration)
print("Best validation log loss:", xgb_binary.best_score)

Starting XGBoost Binary training...
[0]	validation_0-logloss:0.64688
[25]	validation_0-logloss:0.16561
[50]	validation_0-logloss:0.05727
[75]	validation_0-logloss:0.02675
[100]	validation_0-logloss:0.01579
[125]	validation_0-logloss:0.01186
[150]	validation_0-logloss:0.00948
[175]	validation_0-logloss:0.00770
[200]	validation_0-logloss:0.00661
[225]	validation_0-logloss:0.00587
[250]	validation_0-logloss:0.00540
[275]	validation_0-logloss:0.00504
[300]	validation_0-logloss:0.00472
[325]	validation_0-logloss:0.00453
[350]	validation_0-logloss:0.00435
[375]	validation_0-logloss:0.00425
[400]	validation_0-logloss:0.00415
[425]	validation_0-logloss:0.00408
[450]	validation_0-logloss:0.00402
[475]	validation_0-logloss:0.00397
[499]	validation_0-logloss:0.00393

BINARY TRAINING COMPLETED
Training time: 0.82 minutes
Best iteration: 499
Best validation log loss: 0.00392916774188168


In [12]:
# Predict attack probabilities
y_val_prob = xgb_binary.predict_proba(X_val)[:, 1]

# Default threshold
y_val_pred = (y_val_prob >= 0.5).astype(int)

print("BINARY MODEL - VALIDATION RESULTS")
print("=" * 45)

print("Accuracy :", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred))
print("Recall   :", recall_score(y_val, y_val_pred))
print("F1 Score :", f1_score(y_val, y_val_pred))
print("ROC-AUC  :", roc_auc_score(y_val, y_val_prob))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred,
    target_names=["BENIGN", "ATTACK"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

BINARY MODEL - VALIDATION RESULTS
Accuracy : 0.9989223738488994
Precision: 0.9940642186112453
Recall   : 0.9995926807870661
F1 Score : 0.9968207844148135
ROC-AUC  : 0.9999676345174684

Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00    313850
      ATTACK       0.99      1.00      1.00     63832

    accuracy                           1.00    377682
   macro avg       1.00      1.00      1.00    377682
weighted avg       1.00      1.00      1.00    377682

Confusion Matrix:
[[313469    381]
 [    26  63806]]


In [13]:
# Predict probabilities on final test data
y_test_prob = xgb_binary.predict_proba(X_test)[:, 1]

# Apply threshold
y_test_pred = (y_test_prob >= 0.5).astype(int)

print("BINARY MODEL - FINAL TEST RESULTS")
print("=" * 45)

print("Accuracy :", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall   :", recall_score(y_test, y_test_pred))
print("F1 Score :", f1_score(y_test, y_test_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_test_prob))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["BENIGN", "ATTACK"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

BINARY MODEL - FINAL TEST RESULTS
Accuracy : 0.9989859220563276
Precision: 0.9945131170794819
Recall   : 0.9995143501691941
F1 Score : 0.9970074618119311
ROC-AUC  : 0.999963185275976

Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00    313851
      ATTACK       0.99      1.00      1.00     63832

    accuracy                           1.00    377683
   macro avg       1.00      1.00      1.00    377683
weighted avg       1.00      1.00      1.00    377683

Confusion Matrix:
[[313499    352]
 [    31  63801]]


In [14]:
# Create model directory
MODEL_DIR = Path("../models/xgboost_binary")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save trained XGBoost model
model_path = MODEL_DIR / "xgboost_binary.json"
xgb_binary.save_model(model_path)

# Save feature column order
with open(MODEL_DIR / "feature_columns.json", "w") as f:
    json.dump(feature_columns, f, indent=4)

# Save model metadata
metadata = {
    "model_name": "XGBoost Binary Classifier",
    "model_type": "binary:logistic",
    "classes": {
        "0": "BENIGN",
        "1": "ATTACK"
    },
    "threshold": 0.5,
    "number_of_features": len(feature_columns),
    "test_accuracy": float(accuracy_score(y_test, y_test_pred)),
    "test_precision": float(precision_score(y_test, y_test_pred)),
    "test_recall": float(recall_score(y_test, y_test_pred)),
    "test_f1": float(f1_score(y_test, y_test_pred)),
    "test_roc_auc": float(roc_auc_score(y_test, y_test_prob))
}

with open(MODEL_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("BINARY MODEL SAVED SUCCESSFULLY")
print("=" * 40)
print("Model:", model_path)
print("Feature columns: feature_columns.json")
print("Metadata: metadata.json")
print("Files:", [p.name for p in MODEL_DIR.iterdir()])

BINARY MODEL SAVED SUCCESSFULLY
Model: ..\models\xgboost_binary\xgboost_binary.json
Feature columns: feature_columns.json
Metadata: metadata.json
Files: ['.gitkeep', 'feature_columns.json', 'metadata.json', 'xgboost_binary.json']


In [15]:
# Load saved model
loaded_binary_model = XGBClassifier()
loaded_binary_model.load_model(
    MODEL_DIR / "xgboost_binary.json"
)

# Generate predictions using loaded model
loaded_predictions = loaded_binary_model.predict(X_test)

# Compare predictions
same_predictions = np.array_equal(
    y_test_pred,
    loaded_predictions
)

print("MODEL VERIFICATION")
print("=" * 40)
print("Original predictions :", len(y_test_pred))
print("Loaded predictions   :", len(loaded_predictions))
print("Predictions identical:", same_predictions)

if same_predictions:
    print("\nModel verification successful!")
else:
    print("\nModel verification failed!")

MODEL VERIFICATION
Original predictions : 377683
Loaded predictions   : 377683
Predictions identical: True

Model verification successful!
